# 03 — Настройка трекера ByteTrack и расчёт трекинговых метрик

**Цель:** подобрать параметры ByteTrack (track_activation_threshold, minimum_matching_threshold, lost_track_buffer)
на датасете MOT20 (train-последовательности).

**Метрики этапа:** IDF1, MOTA. **Целевой IDF1 ≥ 75%.**

**Входные данные:** веса детектора из `models/` (дообученные на MOT20).

**ByteTrack не обучается** — только параметрически настраивается.

## 1. Импорты и конфигурация

In [ ]:
import sys
import itertools
import json
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np

if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)

import pandas as pd
import matplotlib.pyplot as plt
import motmetrics as mm
import supervision as sv
from ultralytics import YOLO
from tqdm.notebook import tqdm

# ── Пути ──────────────────────────────────────────────────────────────────────
ROOT          = Path().resolve().parent
MOT20_DIR     = ROOT / 'data' / 'MOT20' / 'MOT20'
MODELS_DIR    = ROOT / 'models'
RUNS_DIR      = ROOT / 'runs'

# ── Веса детектора ────────────────────────────────────────────────────────────
MODEL_WEIGHTS = MODELS_DIR / 'yolo11n_mot20_v2.pt'

# ── Параметры детектора (не меняются при настройке трекера) ────────────────────
DET_CONF      = 0.25
DET_IOU       = 0.45
DET_IMGSZ     = 640
DEVICE        = 'cuda'
FRAME_RATE    = 25

# ── Последовательности MOT20 для оценки ──────────────────────────────────────
EVAL_SEQUENCES_GRID  = ['MOT20-01']                                    
EVAL_SEQUENCES_FINAL = ['MOT20-01', 'MOT20-02', 'MOT20-03', 'MOT20-05']  
MAX_FRAMES_GRID      = 300   

# ── Целевая метрика ───────────────────────────────────────────────────────────
TARGET_IDF1 = 0.75

print(f'ROOT:          {ROOT}')
print(f'MODEL_WEIGHTS: {MODEL_WEIGHTS}')
print(f'MOT20_DIR:     {MOT20_DIR}')
print(f'DEVICE:        {DEVICE}')

## 2. Проверка окружения и данных

In [ ]:
import ultralytics

print(f'Python:       {sys.version.split()[0]}')
print(f'PyTorch:      {__import__("torch").__version__}')
print(f'CUDA:         {__import__("torch").cuda.is_available()} / {__import__("torch").version.cuda}')
print(f'GPU:          {__import__("torch").cuda.get_device_name(0)}')
print(f'Ultralytics:  {ultralytics.__version__}')
print(f'Supervision:  {sv.__version__}')
print(f'motmetrics:   {mm.__version__}')
print(f'OpenCV:       {cv2.__version__}')

assert MOT20_DIR.exists(), f'MOT20 не найден: {MOT20_DIR}'
assert MODEL_WEIGHTS.exists(), f'Веса не найдены: {MODEL_WEIGHTS}'

train_seqs = sorted((MOT20_DIR / 'train').iterdir())
print(f'\nMOT20 train последовательности: {[s.name for s in train_seqs]}')

for seq in EVAL_SEQUENCES_FINAL:
    gt = MOT20_DIR / 'train' / seq / 'gt' / 'gt.txt'
    n_frames = len(list((MOT20_DIR / 'train' / seq / 'img1').glob('*.jpg')))
    print(f'  {seq}: {n_frames} кадров, GT {"✅" if gt.exists() else "❌"}')

## 3. Вспомогательные функции: загрузка GT, запуск трекинга, расчёт метрик

In [ ]:
def load_mot_gt(gt_path: Path) -> dict:
    """Загружает ground truth MOT20 в формат {frame_id: [(track_id, x, y, w, h), ...]}."""
    gt = {}
    for line in gt_path.read_text().strip().splitlines():
        p = line.strip().split(',')
        fid, tid = int(p[0]), int(p[1])
        x, y, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
        conf, cls = int(p[6]), int(p[7])
        vis = float(p[8]) if len(p) > 8 else 1.0
        if conf == 0 or cls != 1 or vis < 0.25:
            continue
        gt.setdefault(fid, []).append((tid, x, y, w, h))
    return gt


def run_tracker_on_sequence(model, seq_dir, tracker, max_frames=None,
                             det_conf=DET_CONF, det_iou=DET_IOU, imgsz=DET_IMGSZ):
    """Прогоняет детектор + трекер по кадрам последовательности.
    Возвращает {frame_id: [(track_id, x, y, w, h), ...]}."""
    frames = sorted((seq_dir / 'img1').glob('*.jpg'))
    if max_frames:
        frames = frames[:max_frames]
    preds = {}
    for ff in frames:
        fid = int(ff.stem)
        frame = cv2.imread(str(ff))
        r = model.predict(source=frame, imgsz=imgsz, conf=det_conf,
                          iou=det_iou, device=DEVICE, verbose=False, classes=[0])[0]
        if r.boxes is not None and len(r.boxes):
            dets = sv.Detections(
                xyxy=r.boxes.xyxy.cpu().numpy(),
                confidence=r.boxes.conf.cpu().numpy(),
                class_id=np.zeros(len(r.boxes), dtype=int),
            )
            tracked = tracker.update_with_detections(dets)
        else:
            tracker.update_with_detections(sv.Detections.empty())
            tracked = sv.Detections.empty()
        fp = []
        if tracked.tracker_id is not None:
            for i in range(len(tracked.xyxy)):
                x1, y1, x2, y2 = tracked.xyxy[i]
                fp.append((int(tracked.tracker_id[i]),
                           float(x1), float(y1), float(x2 - x1), float(y2 - y1)))
        preds[fid] = fp
    return preds


def compute_mot_metrics(gt: dict, pred: dict) -> dict:
    """Считает IDF1, MOTA, MOTP, IDSW, Precision, Recall через motmetrics."""
    acc = mm.MOTAccumulator(auto_id=True)
    for fid in sorted(set(gt) | set(pred)):
        gt_o = gt.get(fid, [])
        hyp_o = pred.get(fid, [])
        gt_ids = [o[0] for o in gt_o]
        hyp_ids = [o[0] for o in hyp_o]
        if gt_o and hyp_o:
            gb = np.array([[o[1], o[2], o[3], o[4]] for o in gt_o])
            hb = np.array([[o[1], o[2], o[3], o[4]] for o in hyp_o])
            dist = mm.distances.iou_matrix(gb, hb, max_iou=0.5)
        else:
            dist = mm.distances.iou_matrix(
                np.empty((0, 4)), np.empty((0, 4)), max_iou=0.5
            ) if not gt_o and not hyp_o else None
        if dist is not None:
            acc.update(gt_ids, hyp_ids, dist)
        else:
            acc.update(gt_ids, hyp_ids, [])
    mh = mm.metrics.create()
    s = mh.compute(acc, metrics=['idf1', 'mota', 'motp', 'num_switches',
                                  'precision', 'recall'], name='e')
    return {
        'idf1': float(s['idf1'].iloc[0]),
        'mota': float(s['mota'].iloc[0]),
        'motp': float(s['motp'].iloc[0]),
        'idsw': int(s['num_switches'].iloc[0]),
        'prec': float(s['precision'].iloc[0]),
        'rec':  float(s['recall'].iloc[0]),
    }

print('✅ Функции определены: load_mot_gt, run_tracker_on_sequence, compute_mot_metrics')

## 4. Загрузка модели и базовая оценка (дефолтные параметры ByteTrack)

In [ ]:
model = YOLO(str(MODEL_WEIGHTS))
model.to(DEVICE)
print(f'Модель загружена: {MODEL_WEIGHTS.name}')

print('\n── Базовая оценка (дефолтные параметры) на MOT20-01 ──')
tracker_default = sv.ByteTrack(
    track_activation_threshold=0.25,
    minimum_matching_threshold=0.8,
    lost_track_buffer=30,
    frame_rate=FRAME_RATE,
)

seq_dir = MOT20_DIR / 'train' / 'MOT20-01'
gt_base = load_mot_gt(seq_dir / 'gt' / 'gt.txt')
pred_base = run_tracker_on_sequence(model, seq_dir, tracker_default)
m_base = compute_mot_metrics(gt_base, pred_base)

print(f"  IDF1:      {m_base['idf1']:.2%}")
print(f"  MOTA:      {m_base['mota']:.2%}")
print(f"  IDSW:      {m_base['idsw']}")
print(f"  Precision: {m_base['prec']:.2%}")
print(f"  Recall:    {m_base['rec']:.2%}")

## 5. Grid Search по параметрам ByteTrack

In [ ]:
GRID = {
    'track_activation_threshold': [0.20, 0.25, 0.30],
    'minimum_matching_threshold': [0.7, 0.8, 0.85, 0.9],
    'lost_track_buffer':          [20, 30, 50, 75],
}

gt_grid = {k: v for k, v in gt_base.items() if k <= MAX_FRAMES_GRID}

combos = list(itertools.product(*GRID.values()))
keys = list(GRID.keys())
print(f'Grid Search: {len(combos)} комбинаций, первые {MAX_FRAMES_GRID} кадров MOT20-01')

grid_results = []
for combo in tqdm(combos, desc='Grid Search'):
    kw = dict(zip(keys, combo))
    t = sv.ByteTrack(**kw, frame_rate=FRAME_RATE)
    p = run_tracker_on_sequence(model, seq_dir, t, max_frames=MAX_FRAMES_GRID)
    m = compute_mot_metrics(gt_grid, p)
    grid_results.append({**kw, **m})

df_grid = pd.DataFrame(grid_results).sort_values('idf1', ascending=False)

print('\nТоп-10 по IDF1:')
print(df_grid[['track_activation_threshold', 'minimum_matching_threshold',
               'lost_track_buffer', 'idf1', 'mota', 'idsw']].head(10).to_string(index=False))

## 6. Финальная оценка с лучшими параметрами на всех train-последовательностях

In [ ]:
best_row = df_grid.iloc[0]
BEST_TRACK_THRESH = float(best_row['track_activation_threshold'])
BEST_MATCH_THRESH = float(best_row['minimum_matching_threshold'])
BEST_TRACK_BUFFER = int(best_row['lost_track_buffer'])

print(f'Лучшие параметры (по IDF1 на первых {MAX_FRAMES_GRID} кадрах MOT20-01):')
print(f'  track_activation_threshold = {BEST_TRACK_THRESH}')
print(f'  minimum_matching_threshold = {BEST_MATCH_THRESH}')
print(f'  lost_track_buffer          = {BEST_TRACK_BUFFER}')
print(f'  IDF1 (grid) = {best_row["idf1"]:.2%}')

print(f'\n── Финальная оценка на {len(EVAL_SEQUENCES_FINAL)} последовательностях MOT20 ──')

final_results = []
for seq_name in tqdm(EVAL_SEQUENCES_FINAL, desc='Sequences'):
    s = MOT20_DIR / 'train' / seq_name
    tracker = sv.ByteTrack(
        track_activation_threshold=BEST_TRACK_THRESH,
        minimum_matching_threshold=BEST_MATCH_THRESH,
        lost_track_buffer=BEST_TRACK_BUFFER,
        frame_rate=FRAME_RATE,
    )
    gt_seq = load_mot_gt(s / 'gt' / 'gt.txt')
    pred_seq = run_tracker_on_sequence(model, s, tracker)
    m = compute_mot_metrics(gt_seq, pred_seq)
    final_results.append({'sequence': seq_name, **m})
    print(f'  {seq_name}: IDF1={m["idf1"]:.2%}  MOTA={m["mota"]:.2%}  IDSW={m["idsw"]}')

df_final = pd.DataFrame(final_results)
mean_idf1 = df_final['idf1'].mean()
mean_mota = df_final['mota'].mean()

print(f'\n── Среднее по {len(EVAL_SEQUENCES_FINAL)} последовательностям ──')
print(f'  IDF1 (mean): {mean_idf1:.2%}  (цель ≥ 75%)  {"✅" if mean_idf1 >= TARGET_IDF1 else "❌"}')
print(f'  MOTA (mean): {mean_mota:.2%}')


## 7. Визуализация — пример трека на кадрах MOT20-01

In [ ]:
import random as rnd
rnd.seed(42)

seq_viz_dir = MOT20_DIR / 'train' / 'MOT20-01' / 'img1'
frame_files = sorted(seq_viz_dir.glob('*.jpg'))
sample_frames = rnd.sample(frame_files[50:250], 4)
sample_frames.sort()

tracker_viz = sv.ByteTrack(
    track_activation_threshold=BEST_TRACK_THRESH,
    minimum_matching_threshold=BEST_MATCH_THRESH,
    lost_track_buffer=BEST_TRACK_BUFFER,
    frame_rate=FRAME_RATE,
)

max_sample_idx = int(sample_frames[-1].stem)
frame_viz_preds = {}

for ff in sorted(frame_files)[:max_sample_idx]:
    fid = int(ff.stem)
    frame = cv2.imread(str(ff))
    r = model.predict(source=frame, imgsz=DET_IMGSZ, conf=DET_CONF,
                      iou=DET_IOU, device=DEVICE, verbose=False, classes=[0])[0]
    if r.boxes is not None and len(r.boxes):
        dets = sv.Detections(
            xyxy=r.boxes.xyxy.cpu().numpy(),
            confidence=r.boxes.conf.cpu().numpy(),
            class_id=np.zeros(len(r.boxes), dtype=int),
        )
        tracked = tracker_viz.update_with_detections(dets)
    else:
        tracker_viz.update_with_detections(sv.Detections.empty())
        tracked = sv.Detections.empty()
    if fid in [int(sf.stem) for sf in sample_frames]:
        frame_viz_preds[fid] = (frame, tracked)

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, sf in zip(axes, sample_frames):
    fid = int(sf.stem)
    if fid not in frame_viz_preds:
        ax.axis('off')
        continue
    frame_bgr, tracked = frame_viz_preds[fid]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    if tracked.tracker_id is not None and len(tracked.xyxy):
        labels = [f'ID:{tid}' for tid in tracked.tracker_id]
        frame_rgb = box_annotator.annotate(frame_rgb, tracked)
        frame_rgb = label_annotator.annotate(frame_rgb, tracked, labels)
    ax.imshow(frame_rgb)
    ax.set_title(f'Кадр {fid}  |  треков: {len(tracked.xyxy) if tracked.tracker_id is not None else 0}', fontsize=10)
    ax.axis('off')

plt.suptitle('ByteTrack — треки на MOT20-01', fontsize=13)
plt.tight_layout()
plt.savefig(ROOT / 'runs' / 'tracker_vis.png', dpi=100, bbox_inches='tight')
plt.show()
print('Визуализация сохранена: runs/tracker_vis.png')


## 8. Итоговая сводка

In [ ]:
print('=' * 60)
print('   ИТОГОВАЯ СВОДКА — 03_tracker_tuning_eval')
print('=' * 60)

print('\nЛучшие параметры ByteTrack:')
print(f'  track_activation_threshold = {BEST_TRACK_THRESH}')
print(f'  minimum_matching_threshold = {BEST_MATCH_THRESH}')
print(f'  lost_track_buffer          = {BEST_TRACK_BUFFER}')

print(f'\nМетрики по последовательностям MOT20:')
print(df_final[['sequence', 'idf1', 'mota', 'motp', 'idsw', 'prec', 'rec']].to_string(index=False))

print(f'\nСреднее:')
print(f'  IDF1 = {mean_idf1:.2%}  (цель >= 75%)  {"✅" if mean_idf1 >= TARGET_IDF1 else "❌"}')
print(f'  MOTA = {mean_mota:.2%}')

if mean_idf1 >= TARGET_IDF1:
    print('\n✅ Цель IDF1 >= 75% достигнута!')
    print('   Следующий шаг -> 04_metrics_acceptance.ipynb')
else:
    gap = TARGET_IDF1 - mean_idf1
    print(f'\n⚠  IDF1 = {mean_idf1:.2%} — не хватает {gap:.2%} до цели.')
    print('   Рекомендации:')
    print('   1. Увеличить conf threshold детектора (уменьшить FP)')
    print('   2. Попробовать track_buffer > 75 для длинных окклюзий')
    print('   3. Дообучить детектор на специфичных сценах')

print('=' * 60)

best_params = {
    'track_activation_threshold': BEST_TRACK_THRESH,
    'minimum_matching_threshold': BEST_MATCH_THRESH,
    'lost_track_buffer':          BEST_TRACK_BUFFER,
    'frame_rate':                 FRAME_RATE,
    'det_conf':                   DET_CONF,
    'det_iou':                    DET_IOU,
    'model_weights':              str(MODEL_WEIGHTS.relative_to(ROOT)),
    'idf1_mot20_train':           round(mean_idf1, 4),
    'mota_mot20_train':           round(mean_mota, 4),
}
best_params_path = MODELS_DIR / 'bytetrack_best_params.json'
with open(best_params_path, 'w') as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)
print(f'\n✅ Параметры сохранены: {best_params_path}')
